# Risk indicators — quickstart (live)

The `risk-indicators` backend pulls **country/admin-indexed risk
screening** from three sources behind one interface:

- **GFDRR ThinkHazard!** — a hazard rating (Very low → High) per hazard
  type and admin division; **public**, no key.
- **INFORM Risk (JRC)** — the composite humanitarian-risk index and its
  sub-dimensions, scored 0–10 per country; **public**, no key.
- **Global Forest Watch** Data API — forest indicators and admin geometry;
  needs a free **API key** (covered in the
  [Global Forest Watch notebook](03_global_forest_watch.ipynb)).

Unlike the raster backends there is no bounding box or grid — you name a
dataset id in `variables=` and a country with `country=` (an ISO3 code),
and you get back a tidy `pandas.DataFrame`. This notebook covers the two
**public** sources (ThinkHazard! and INFORM), fetched **live**.

In [ ]:
import tempfile

from earthlens.core import EarthLens

OUT = tempfile.mkdtemp()  # tabular downloads also write a CSV here

## ThinkHazard! — one hazard for a country

`variables=["thinkhazard:flood_river"]` selects the river-flood rating;
`country="KEN"` names the country as an ISO3 code. The backend resolves
that ISO3 to ThinkHazard's numeric ADM0 division code and returns one row
with the qualitative `level` / `level_title`.

In [ ]:
flood = EarthLens(
    data_source="risk-indicators",
    variables=["thinkhazard:flood_river"],
    country="KEN",
    path=OUT,
).download()

flood

The `level` is ThinkHazard's coded rating (`VLO`/`LOW`/`MED`/`HIG`) and
`level_title` its human label; `admin_code` is the resolved ADM0 code
(`133` for Kenya). `download()` also wrote the row to a CSV under `path`.

## All eleven hazards at once

The `thinkhazard:all` dataset returns every hazard ThinkHazard! screens
for the division in a single call — handy for a quick risk profile.

In [ ]:
profile = EarthLens(
    data_source="risk-indicators",
    variables=["thinkhazard:all"],
    country="KEN",
    path=OUT,
).download()

print("hazards:", len(profile))
profile

## INFORM Risk — a composite country score

`inform:risk` is the headline INFORM composite index (0–10). The
`"inform"` alias and the `"risk-indicators"` key both reach the same
backend, so either works as `data_source=`.

The first call downloads JRC's release workbook (a few MB) and caches
it, so the later INFORM cells reuse it rather than re-downloading.

In [ ]:
risk = EarthLens(
    data_source="inform",
    variables=["inform:risk"],
    country="KEN",
    path=OUT,
).download()

risk

One row per country: `indicator_score` is the 0–10 composite (higher =
more at risk), `iso3` the country. The last three columns record where the numbers came
from: `source` is `release` (JRC's published workbook, the default and the
current release) or `api`, `workflow_id` names the model release an API row
used, and `validity_year` carries the workbook's release year — the API
leaves it at `0`. Pass `source="api"` or `workflow_id=` to switch.

**Omit `country=` and INFORM returns every country** — the whole global
table in one frame, ready to rank or join.

In [ ]:
world = EarthLens(
    data_source="inform",
    variables=["inform:risk"],
    path=OUT,
).download()

print("countries:", len(world))
world.sort_values("indicator_score", ascending=False).head(5)

## The `country=` selector

`country=` is always an **ISO3** code (`"KEN"`, `"NGA"`, `"BRA"`, …).
What happens to it depends on the source:

- **ThinkHazard!** resolves the ISO3 to a GAUL 2015 **ADM0 division
  code** (the numeric key its API uses) via the catalog's bundled
  `admin_codes:` table. For a sub-national division, pass a raw
  `admin_code=` instead.
- **INFORM** filters its global table to that one ISO3 (omit it for all
  countries, as above).
- **GFW** uses the ISO3 as the SQL / geostore key (see the
  [GFW notebook](03_global_forest_watch.ipynb)).

## Takeaway

- `EarthLens(data_source="risk-indicators", variables=[<id>],
  country=<ISO3>)` returns a tidy `DataFrame` — no bbox, no grid.
- ThinkHazard! gives qualitative hazard ratings per division
  (`thinkhazard:all` for the full profile); INFORM gives 0–10 composite
  scores (omit `country=` for the whole world).
- Both are **public** and keyless. The third source, Global Forest Watch,
  needs a free key — see the
  [catalog & behaviour](02_catalog_and_behavior.ipynb) notebook for the
  full dataset list and the [GFW notebook](03_global_forest_watch.ipynb)
  for the keyed forest indicators.